In [0]:
from pyspark.sql import Row
import random

random.seed(42)

metiers = ["Data Engineer", "ML Engineer", "Data Scientist", "AI Engineer", "Data Analyst"]
villes = ["Paris", "Lyon", "Bordeaux", "Toulouse", "Nantes", "Lille", "Marseille"]
entreprises = ["Capgemini", "Thales", "BNP Paribas", "Orange", "Sopra Steria", "Amadeus", "Dassault"]

data = [
    Row(
        titre=random.choice(metiers),
        ville=random.choice(villes),
        salaire=random.randint(35000, 95000),
        entreprise=random.choice(entreprises),
        annee_experience=random.randint(0, 10)
    )
    for _ in range(1500)
]

df_spark = spark.createDataFrame(data)
print(f"✅ Données créées : {df_spark.count()} lignes")
df_spark.show(5)

✅ Données créées : 1500 lignes
+-------------+-----+-------+-----------+----------------+
|        titre|ville|salaire| entreprise|annee_experience|
+-------------+-----+-------+-----------+----------------+
|Data Engineer|Paris|  83598|BNP Paribas|               3|
|  ML Engineer| Lyon|  83265|  Capgemini|              10|
| Data Analyst|Paris|  73698|     Orange|               0|
|Data Engineer|Paris|  49328|     Thales|               8|
| Data Analyst|Paris|  71781|     Thales|              10|
+-------------+-----+-------+-----------+----------------+
only showing top 5 rows


In [0]:
from pyspark.sql.functions import col

df_clean = df_spark.dropDuplicates().dropna()
df_clean = df_clean.filter((col("salaire") >= 30000) & (col("salaire") <= 100000))

print(f"✅ Après nettoyage : {df_clean.count()} lignes")

✅ Après nettoyage : 1500 lignes


In [0]:
df_clean.write.format("delta").mode("overwrite").saveAsTable("offres_ia_france")

print("✅ Table Delta créée : offres_ia_france")

✅ Table Delta créée : offres_ia_france


In [0]:
%sql
SELECT titre, ROUND(AVG(salaire), 0) AS salaire_moyen
FROM offres_ia_france
GROUP BY titre
ORDER BY salaire_moyen DESC

titre,salaire_moyen
Data Engineer,66176.0
ML Engineer,65749.0
AI Engineer,65017.0
Data Scientist,64900.0
Data Analyst,64704.0


In [0]:
%sql
SELECT ville, ROUND(AVG(salaire), 0) AS salaire_moyen
FROM offres_ia_france
GROUP BY ville
ORDER BY salaire_moyen DESC

ville,salaire_moyen
Lyon,65874.0
Marseille,65735.0
Paris,65392.0
Toulouse,65312.0
Lille,65298.0
Bordeaux,64754.0
Nantes,64719.0


In [0]:
%sql
SELECT entreprise, COUNT(*) AS nb_offres
FROM offres_ia_france
GROUP BY entreprise
ORDER BY nb_offres DESC

entreprise,nb_offres
Sopra Steria,236
Amadeus,230
Thales,223
Dassault,217
BNP Paribas,203
Orange,201
Capgemini,190
